
Smart Banking Support Assistant_ without API.ipynb
Smart Banking Support Assistant_ without API.ipynb_
MCP SCENARIO: “Smart Banking Support Assistant” 🧩 Scenario Background You are working in a company called FinTrust Bank. Customers often face issues such as:

Credit card not working
Trouble with online banking login
Queries about loan status
Transaction disputes 👉 Instead of calling customer care, customers use an AI Banking Support Bot.
🤖 What this Bot Should Do When a customer types a problem:

Understand the issue (e.g., “My card was declined”)
Decide if escalation to a human agent is needed
Identify:
Category (Card Services / Online Banking / Loans / Transactions)
Priority (High / Medium)
Create a support ticket if required
Provide instant guidance (FAQs, troubleshooting steps, policy info)
Show confirmation and next steps
🧠 How MCP Fits Here | | | | | | | | | | | | | | |

This way, MCP is applied in a financial services context, where the AI assistant reduces call center load, provides quick resolutions, and ensures customers feel supported with secure, reliable guidance. Would you like me to craft one more in a healthcare setting (like hospital patient support), so you can see how MCP adapts to critical service environments?


[ ]
1
Colab paid products - Cancel contracts here
Hello, MRINAL
How can I help you today?
What can I help you build?
Gemini 2.5 Flash


In [ ]:
!pip install -q groq

import os
import json
from datetime import datetime
from groq import Groq
from google.colab import userdata

# ============================================
# STEP 0: CLIENT SETUP
# ============================================

api_key = userdata.get("GROQ_API_KEY")
client = Groq(api_key=api_key)

# ============================================
# STEP 1: DATABASE (Simulated storage)
# ============================================

tickets_db = []

# ============================================
# STEP 2: TOOL LAYER
# ============================================

def create_support_ticket(issue, priority, category, context):
    """
    MCP Tool:
    Simulates banking support ticket creation.
    In real-world, this could connect to CRM / ticketing / banking support systems.
    """
    ticket_id = f"BNK{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category,
        "created_at": datetime.now().isoformat(),
        "source": "banking_support_bot",
        "context": context
    }

    tickets_db.append(ticket)
    return ticket

# ============================================
# STEP 3: CONTEXT OBJECT
# ============================================

def build_context(user_input):
    """
    MCP-style context object.
    """
    return {
        "user_input": user_input,
        "session_id": f"session_{len(tickets_db) + 1}",
        "timestamp": datetime.now().isoformat(),
        "agent_name": "BankingSupportAgent"
    }

# ============================================
# STEP 4: GUIDANCE LAYER
# ============================================

def provide_guidance(category):
    faq_map = {
        "card services": (
            "Try these steps:\n"
            "1. Check if your card is active and not expired\n"
            "2. Verify you entered the correct PIN/OTP\n"
            "3. Check if international or online usage is enabled\n"
            "4. If the card is still declined, support escalation is needed"
        ),
        "online banking": (
            "Try these steps:\n"
            "1. Confirm your username and password\n"
            "2. Use 'Forgot Password' if login fails\n"
            "3. Check if your mobile number is linked for OTP verification\n"
            "4. If access is blocked, banking support is needed"
        ),
        "loans": (
            "Try these steps:\n"
            "1. Check your loan application/reference number\n"
            "2. Review loan status in the official portal/app\n"
            "3. Keep ID and application details ready\n"
            "4. If status is unclear or delayed, support can assist"
        ),
        "transactions": (
            "Try these steps:\n"
            "1. Review recent transaction history\n"
            "2. Confirm if the amount is pending or posted\n"
            "3. Check merchant details and SMS/email alerts\n"
            "4. If the transaction looks incorrect, raise a dispute ticket"
        ),
        "general": (
            "Please share a bit more detail about your banking issue so I can classify it correctly."
        )
    }
    return faq_map.get(category, faq_map["general"])

# ============================================
# STEP 5: LLM ANALYSIS LAYER
# ============================================

def analyze_with_llm(user_input, context):
    """
    LLM decides:
    - create_ticket
    - category
    - priority
    - short_reason
    """

    prompt = f"""
You are a banking support assistant for FinTrust Bank.

Analyze the customer's issue and return ONLY valid JSON.

Rules:
- create_ticket = true if the issue clearly needs escalation to a human support agent
- create_ticket = false if the issue is only informational or can be solved with basic guidance
- category must be one of:
  "card services", "online banking", "loans", "transactions", "general"
- priority must be one of:
  "high", "medium"

Return exactly this JSON schema:
{{
  "create_ticket": true,
  "category": "card services",
  "priority": "high",
  "short_reason": "Customer's card is declined and payment is blocked"
}}

Context:
{json.dumps(context, indent=2)}

User Input:
"{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "Return only valid JSON. No markdown. No explanation."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    output = response.choices[0].message.content.strip()

    try:
        parsed = json.loads(output)
    except Exception:
        try:
            start = output.find("{")
            end = output.rfind("}") + 1
            parsed = json.loads(output[start:end])
        except Exception:
            parsed = {
                "create_ticket": True,
                "category": "general",
                "priority": "medium",
                "short_reason": "Fallback used due to JSON parse issue"
            }

    parsed["category"] = str(parsed.get("category", "general")).lower().strip()
    parsed["priority"] = str(parsed.get("priority", "medium")).lower().strip()

    if parsed["category"] not in {"card services", "online banking", "loans", "transactions", "general"}:
        parsed["category"] = "general"

    if parsed["priority"] not in {"high", "medium"}:
        parsed["priority"] = "medium"

    parsed["create_ticket"] = bool(parsed.get("create_ticket", True))
    parsed["short_reason"] = parsed.get("short_reason", "No reason provided")

    return parsed

# ============================================
# STEP 6: EXECUTION METADATA
# ============================================

def build_metadata(context, decision):
    return {
        "agent_name": context["agent_name"],
        "session_id": context["session_id"],
        "timestamp": datetime.now().isoformat(),
        "decision_summary": decision["short_reason"]
    }

# ============================================
# STEP 7: MCP ORCHESTRATOR
# ============================================

def mcp_banking_agent(user_input):
    """
    Main MCP flow:
    User -> Context -> LLM Analysis -> Decision -> Tool Call -> Final Response
    """

    context = build_context(user_input)
    print("\n🧠 Agent received:", user_input)
    print("🗂️ Context:", context)

    decision = analyze_with_llm(user_input, context)
    print("🤖 LLM Decision:", decision)

    guidance = provide_guidance(decision["category"])

    metadata = build_metadata(context, decision)
    print("📝 Metadata:", metadata)

    if decision["create_ticket"]:
        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"],
            "context": context
        }

        print("📦 MCP Payload:", payload)

        result = create_support_ticket(**payload)

        return f"""
✅ Support Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
Created At: {result['created_at']}

Why ticket was created:
- {decision['short_reason']}

Instant Guidance:
{guidance}

Next Step:
- Our banking support team will review this case shortly.
- Keep your ticket ID for future follow-up.
"""

    else:
        return f"""
🤖 No Ticket Required Right Now

Reason:
- {decision['short_reason']}

Suggested Guidance:
{guidance}

Next Step:
- Try the above banking guidance first.
- If the issue continues, raise the issue again with more detail.
"""

# ============================================
# STEP 8: RUN LOOP
# ============================================

print("🚀 LLM + MCP-style Banking Support Assistant Started (type 'exit')\n")

while True:
    user_input = input("Enter banking issue: ").strip()

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    if not user_input:
        print("⚠️ Please enter a valid issue.")
        continue

    response = mcp_banking_agent(user_input)
    print(response)

🚀 LLM + MCP-style Banking Support Assistant Started (type 'exit')

Enter banking issue: My credit card was declined and this is urgent

🧠 Agent received: My credit card was declined and this is urgent
🗂️ Context: {'user_input': 'My credit card was declined and this is urgent', 'session_id': 'session_1', 'timestamp': '2026-03-28T06:49:24.509617', 'agent_name': 'BankingSupportAgent'}
🤖 LLM Decision: {'create_ticket': True, 'category': 'card services', 'priority': 'high', 'short_reason': "Customer's credit card is declined"}
📝 Metadata: {'agent_name': 'BankingSupportAgent', 'session_id': 'session_1', 'timestamp': '2026-03-28T06:49:24.820199', 'decision_summary': "Customer's credit card is declined"}
📦 MCP Payload: {'issue': 'My credit card was declined and this is urgent', 'priority': 'high', 'category': 'card services', 'context': {'user_input': 'My credit card was declined and this is urgent', 'session_id': 'session_1', 'timestamp': '2026-03-28T06:49:24.509617', 'agent_name': 'BankingSu